In [ ]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict


In [ ]:
DATA_DIR = "/kaggle/input/stanford-rna-3d-folding-2"
train_sequences = pd.read_csv(f"{DATA_DIR}/train_sequences.csv")
test_sequences = pd.read_csv(f"{DATA_DIR}/test_sequences.csv")
train_labels = pd.read_csv(f"{DATA_DIR}/train_labels.csv", low_memory=False)
print("Train sequences:", len(train_sequences))
print("Test sequences:", len(test_sequences))
print("Labels rows:", len(train_labels))


In [ ]:
sequence_index = dict(zip(train_sequences["target_id"], train_sequences["sequence"]))
reverse_sequence_index = {}
for tid, seq in sequence_index.items():
    reverse_sequence_index[seq] = tid
print("Sequence index built:", len(reverse_sequence_index))


In [ ]:
structure_cache = {}
grouped = train_labels.groupby(train_labels["ID"].str.split("_").str[0])
for target_id, group in grouped:
    coords = group[["x_1","y_1","z_1"]].values
    residues = group["resname"].values
    structure_cache[target_id] = (residues, coords)
print("Structures cached:", len(structure_cache))


In [ ]:
def perturb_structure(coords, scale=0.5):
    noise = np.random.normal(0, scale, coords.shape)
    return coords + noise

def rotate_structure(coords, angle):
    rot = np.array([
        [np.cos(angle), -np.sin(angle), 0],
        [np.sin(angle),  np.cos(angle), 0],
        [0,0,1]
    ])
    return coords @ rot.T

def generate_ensemble(coords):
    candidates = []
    candidates.append(coords)
    candidates.append(perturb_structure(coords))
    candidates.append(perturb_structure(coords, 1.0))
    candidates.append(rotate_structure(coords, 0.1))
    candidates.append(rotate_structure(coords, -0.1))
    return candidates[:5]


In [ ]:
pred_rows = []
fallback_template = list(structure_cache.keys())[0]
for tid, seq in zip(test_sequences.iloc[:,0], test_sequences.iloc[:,1]):
    match = reverse_sequence_index.get(seq)
    if match is None:
        match = fallback_template
    residues, coords = structure_cache.get(match)
    candidates = generate_ensemble(coords)
    for model_idx, candidate in enumerate(candidates):
        for i, (x,y,z) in enumerate(candidate):
            pred_rows.append({
                "ID": f"{tid}_{i+1}",
                "model": model_idx,
                "x": x,
                "y": y,
                "z": z
            })
print("Predictions generated:", len(pred_rows))


In [ ]:
submission = pd.DataFrame(pred_rows)
submission.to_csv("submission.csv", index=False)
print("submission.csv written")
submission.head()
